In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import glob


In [2]:
# Define file path patterns
validation_path = "/kaggle/input/validation-answers/annotated_instances*.csv"
sample_path = "/kaggle/input/emi-speeches-validation-sample/speeches_validation_sample.csv"
speeches_path = "/kaggle/input/final-data/speeches_all_emi.csv"

# Read all validation files
validation_files = glob.glob(validation_path)

df_list = [pd.read_csv(file) for file in validation_files]
df = pd.concat(df_list, ignore_index=True)

# Read the sample and speeches data
sample = pd.read_csv(sample_path)
speeches = pd.read_csv(speeches_path)

# Define valid responses for test questions
valid_intuition_responses = {"Stimme voll und ganz zu", "Stimme zu", "Neutral"}
valid_evidence_responses = {"Stimme voll und ganz zu", "Stimme zu", "Neutral"}

# Identify users who failed the test questions
failed_users = df[
    ((df["instance_id"] == "1_testing") & (~df["Rating:::Intuition basiert"].isin(valid_intuition_responses))) |
    ((df["instance_id"] == "0_testing") & (~df["Rating:::Evidenz basiert"].isin(valid_evidence_responses)))
]["user"].unique()

#df = df[~df["user"].isin(failed_users)]

# Mapping responses to values
response_map = {
    'Stimme überhaupt nicht zu': -2,
    'Stimme nicht zu': -1,
    'Neutral': 0,
    'Stimme zu': 1,
    'Stimme voll und ganz zu': 2
}

df['Evidenz_values'] = df['Rating:::Evidenz basiert'].map(response_map)
df['Intuition_values'] = df['Rating:::Intuition basiert'].map(response_map)

# Group by instance_id and create lists of values
grouped_df = df.groupby('instance_id').agg({
    'Evidenz_values': lambda x: list(x.dropna()),
    'Intuition_values': lambda x: list(x.dropna()),
    'user': 'count'
}).reset_index()

# Rename columns and add the 'N' column
grouped_df.rename(columns={
    'Evidenz_values': 'evidence survey',
    'Intuition_values': 'intuition survey',
    'user': 'N'
}, inplace=True)

grouped_df.rename(columns={'instance_id': 'id'}, inplace=True)

# Merge with sample data
merged_df = grouped_df.merge(
    sample[['id', 'original_text', "text", "emi_bin", "intuition_bin", "evidence_bin", "decade", "year"]],
    on='id',
    how='left'
)

# Ensure both 'id' columns are strings for consistent merging
merged_df['id'] = merged_df['id'].astype(str)
speeches['id'] = speeches['id'].astype(str)

# Merge with speeches data
merged_df = merged_df.merge(
    speeches,
    on=['id', 'text'],  # Use both 'id' and 'text' as unique keys for the merge
    how='left'          # Specify the join type
)

# Save the merged DataFrame
merged_df.to_csv("validation_results.csv", index=False)

print("Merged validation results saved successfully.")

<ipython-input-2-4ca24982e4f1>:14: DtypeWarning: Columns (7,8,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  speeches = pd.read_csv(speeches_path)


Merged validation results saved successfully.


In [3]:
failed_users

array(['6732c3c5979c10e6d55dedc7', '64f1e17e5439f81a8eade6c2',
       '60e9bc26b5ca2c77013810b8', '670a484f463abbed664314a1',
       '60c5f64608278066e948c64b', '66a26c1f25e635ae967419ef',
       '659c10e095ca8d2e4b1d5048', '6740e6fd8d6a08564439533b',
       '6790e4091651fd43b263386d', '665f10dc49cd980fdf304c19',
       '66366b8476401d220319da55', '6661b7808dd1cb864550c052',
       '5b9f7f9eaa0e0e0001573bd2', '67c02ddea53434ef94274b7c',
       '668d35eb0f62647ae823f969', '66604d28d949fc95c6ce4ef1',
       '6525812fe3eb1b505314d6e2', '5e95c2f0818c5d0009101005',
       '67334d4fca6e64a7b62d3f2d', '66e81dc7059677bbb30aff0c',
       '6788f8f6e92c3cd521bf9d67', '6713ff833df413a8e1d83bf8',
       '67b9e921b4b5ee4019522dbb', '6745d8ba436b1416f92aa9b8',
       '67075cb6e533b245cc1e49df', '6452986615d35c79e6ca6665',
       '674602882d723edde43ab53e', '66a23cd771afea46e9f0a3ad',
       '67caec1856007f2748d915d0', '643a2ebe449fcc17e5bc25b9',
       '6676d661aebf60ef65ee1d43', '67263450bf46c38b6cf

In [4]:
merged_df

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pan

,id,evidence survey,intuition survey,N,original_text,text,emi_bin,intuition_bin,evidence_bin,decade,...,evidence_score,intuition_score,chunk_length_bin,evidence_mean,evidence_adj,intuition_mean,intuition_adj,evidence_z,intuition_z,emi
0,0_testing,"[2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 1.0, 0.0, 2.0, ...","[2.0, 1.0, -1.0, 0.0, 2.0, 2.0, -1.0, -1.0, -2...",339,Die Daten unserer umfangreichen Studien zeigen...,Die Daten unserer umfangreichen Studien zeigen...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10001,"[1.0, -2.0, -2.0, 2.0, 2.0]","[-1.0, 2.0, 2.0, 2.0, -2.0]",5,"Ich lehne es ab, mich in diesem Hohen Hause au...",lehne Hohen Hause Niveau begeben Kabarett übli...,"(-5.186191999999999, -2.1966578749999996]","(0.46014310000000025, 2.6364639000000003]","(-1.70590185, 1.1031212999999997]",1950.0,...,-0.023104,0.226178,"(40, 50]",0.044534,-0.067638,0.018643,0.207535,-0.707661,1.780287,-2.487948
2,1004280,"[1.0, 1.0, 1.0, 1.0, -1.0]","[-1.0, 0.0, 2.0, 2.0, 2.0]",5,"Frau Kollegin, vielen Dank, dass Sie noch einm...",Kollegin vielen Dank darauf hinweisen deutlich...,"(0.79287625, 3.7824103750000004]","(0.46014310000000025, 2.6364639000000003]","(1.1031212999999997, 3.9121444499999996]",2010.0,...,0.198081,0.115429,"(30, 40]",-0.009948,0.208029,-0.011461,0.126891,2.176502,1.088502,1.088000
3,1005062,"[1.0, -2.0, -1.0, 2.0, -2.0]","[-2.0, 2.0, 1.0, 1.0, 2.0]",5,"Vielen Dank, Frau Kollegin, dass ich diese Zwi...",Vielen Dank Kollegin Zwischenfrage stellen dar...,"(0.79287625, 3.7824103750000004]","(0.46014310000000025, 2.6364639000000003]","(3.9121444499999996, 6.721167599999999]",2010.0,...,0.476700,0.180839,"(20, 30]",-0.031679,0.508379,-0.020281,0.201120,5.318909,1.725261,3.593648
4,1006590,"[1.0, -2.0, -1.0, -1.0, -2.0]","[2.0, 2.0, 1.0, 1.0, 2.0]",5,Da die Staatsanwaltschaften und Gerichte diese...,Staatsanwaltschaften Gerichte Schlussfolgerung...,"(-2.1966578749999996, 0.79287625]","(0.46014310000000025, 2.6364639000000003]","(-1.70590185, 1.1031212999999997]",2010.0,...,0.107220,0.253291,"(40, 50]",0.044534,0.062686,0.018643,0.234648,0.655849,2.012872,-1.357023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1140,97876,"[0.0, 1.0, 2.0, 0.0, 2.0]","[1.0, -2.0, -2.0, 0.0, 2.0]",5,"Herr Kollege Lohmar, würden Sie nicht doch an ...",Kollege Lohmar Hand zitierten Texte Verwaltung...,"(3.7824103750000004, 6.7719445]","(-1.7161776999999998, 0.46014310000000025]","(3.9121444499999996, 6.721167599999999]",1960.0,...,0.350174,-0.065635,"(20, 30]",-0.031679,0.381853,-0.020281,-0.045353,3.995134,-0.389053,4.384187
1141,9796,"[-2.0, 1.0, -1.0, 1.0, 1.0, -2.0, 1.0, -1.0, 1...","[2.0, -1.0, 1.0, 0.0, 1.0, 2.0, -1.0, 1.0, 0.0...",10,"Es gibt für Ihre Auffassung, Herr Mommer, nur ...",gibt Auffassung Mommer Erklärung nämlich Mitgl...,"(-2.1966578749999996, 0.79287625]","(2.6364639000000003, 4.8127847]","(1.1031212999999997, 3.9121444499999996]",1950.0,...,0.161373,0.357907,"(30, 40]",-0.009948,0.171321,-0.011461,0.369368,1.792441,3.168539,-1.376097
1142,98182,"[0.0, -1.0, 0.0, 2.0, 2.0, 0.0, -1.0, 0.0, 2.0...","[0.0, 1.0, 0.0, 0.0, -2.0, 0.0, 1.0, 0.0, 0.0,...",10,"Herr Dr. Czaja, darf ich Sie bitten, dann auch...",Czaja darf bitten Abs interpretieren heißt Min...,"(0.79287625, 3.7824103750000004]","(-1.7161776999999998, 0.46014310000000025]","(-1.70590185, 1.1031212999999997]",1960.0,...,0.020748,-0.185367,"(10, 20]",-0.031656,0.052404,-0.016343,-0.169024,0.548281,-1.449933,1.998214
1143,98968,"[-1.0, 0.0, 1.0, 1.0, -1.0, 0.0, 1.0, 1.0]","[1.0, 0.0, -1.0, -1.0, 1.0, 0.0, -1.0, -1.0]",8,"Herr Kollege Dr. Burgbacher, kennen Sie nicht ...",Kollege Burgbacher kennen zahlreichen Veröffen...,"(0.79287625, 3.7824103750000004]","(0.46014310000000025, 2.6364639000000003]","(1.1031212999999997, 3.9121444499999996]",1960.0,...,0.235649,0.188002,"(20, 30]",-0.031679,0.267327,-0.020281,0.208284,2.796908,1.786712,1.010196
